# TF-IDF Retrieval

This notebook implements the retrieval component of the retrieval-augmented generation (RAG) pipeline.

## Purpose

TF-IDF is used as a sparse retrieval baseline to retrieve the course-material chunks that are most relevant to a student question. The retrieved chunks can later be passed to an LLM or a sequence-to-sequence model to generate an answer.

## Pipeline

1. Load the preprocessed course-material chunks created in Notebook 03.
2. Build a TF-IDF index from the course-material chunks.
3. Retrieve the top-$k$ chunks for each question using cosine similarity.
4. Evaluate retrieval performance against the annotated source pages.
5. Use the retrieved chunks as context for the LLM and sequence-to-sequence experiments.

- **Train:** retrieved contexts are used as input to the LLM and sequence-to-sequence experiments.
- **Validation:** retrieved contexts are used to tune and evaluate the retrieval and generation pipeline.
- **Test:** retrieved contexts are used for the final, held-out evaluation.

## Data flow

- **Input documents:** `data/processed/chunks_preprocessed.jsonl`
- **Questions:** `data/splits/train.jsonl`, `data/splits/validation.jsonl`, and `data/splits/test.jsonl`
- **Retriever:** TF-IDF with cosine similarity
- **Output:** ranked chunks, retrieved page references, and retrieval metrics

The TF-IDF retriever is fitted on the course-material chunks. The same fitted retriever is then used to retrieve relevant chunks for the train, validation, and test questions.

## RAG Retrieval Pipeline

```text
Notebook 03
Preprocessing and splitting
        |
        v
Preprocessed course chunks
        |
        v
Notebook 04
TF-IDF retriever
        |
        +------------------+
        |                  |
        v                  v
Train questions      Validation questions
retrieved contexts   tune and evaluate
        |
        v
Answer generator
        |
        +------------------+
        |                  |
        v                  v
      LLM              Seq2Seq
        |                  |
        +--------+---------+
                 v
              Answers

Test questions
        |
        v
Final retrieval evaluation
        |
        v
Final LLM and Seq2Seq evaluation
```

In [20]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
PROJECT_ROOT = Path.cwd()

PREPROCESSING_NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "03_proccessingSplitting.ipynb"
)

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks.jsonl"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.jsonl"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.jsonl"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.jsonl"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PREPROCESSING_NOTEBOOK_PATH.exists()
assert CHUNKS_PATH.exists()
assert TRAIN_PATH.exists()
assert VALIDATION_PATH.exists()
assert TEST_PATH.exists()

print("Notebook 03:", PREPROCESSING_NOTEBOOK_PATH)
print("Chunkovi:", CHUNKS_PATH)
print("Trening skup:", TRAIN_PATH)
print("Validacioni skup:", VALIDATION_PATH)
print("Test skup:", TEST_PATH)

Notebook 03: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/03_proccessingSplitting.ipynb
Chunks: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/processed/chunks.jsonl
Train: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/train.jsonl
Validation: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/validation.jsonl
Test: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/test.jsonl


In [ ]:
with PREPROCESSING_NOTEBOOK_PATH.open("r", encoding="utf-8") as file:
    notebook_03 = json.load(file)

load_jsonl_source = next(
    cell["source"]
    for cell in notebook_03["cells"]
    if cell.get("cell_type") == "code"
    and any(
        line.startswith("def load_jsonl")
        for line in cell.get("source", [])
    )
)

exec("".join(load_jsonl_source), globals())
print("Funkcija load_jsonl je učitana iz notebooka 03.")

load_jsonl je učitan iz notebooka 03.


In [ ]:
chunks = load_jsonl(CHUNKS_PATH)

train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"Broj chunkova: {len(chunks)}")
print(f"Broj pitanja u trening skupu: {len(train_data)}")
print(f"Broj pitanja u validacionom skupu: {len(validation_data)}")
print(f"Broj pitanja u test skupu: {len(test_data)}")

Number of chunks: 344
Number of train questions: 100
Number of validation questions: 21
Number of test questions: 22
